# Deep Dive: Agent SDK, Production Agentic Loops & Custom Tools

## 📋 Summary

An agent is not a mysterious black box — it is a **while-loop around an LLM with tools**, governed by a strict contract. High-level frameworks like the **Claude Agent SDK** (`query()` + `allowedTools` + custom MCP tools) automate that loop for you, while knowing how to build the loop from scratch lets you enforce budgets, audit traces, and clean error recovery.

In this notebook (covering **Lessons 05–06**: Agent SDK, Python Agentic Loops & Audit/Decomposition, and Custom Tools), you will build a **Personal Finance & Expense Auditor** agent from first principles:
- Control flow driven strictly by `response.stop_reason`
- Token + wall-clock `Budget` enforcement (raising `BudgetExceeded`, never silently truncating)
- Structured custom tool schemas with `is_error: True` error handling
- Audit tracing and dynamic multi-step decomposition

### Key concepts in this notebook

| # | Concept | Complexity |
|---|---|---|
| 1 | Anatomy of a custom tool definition (name, description, `input_schema`) | 🟢 Beginner |
| 2 | The `stop_reason` contract (`tool_use` vs `end_turn` vs unexpected) | 🟢 Beginner |
| 3 | Returning tool errors as `is_error: True` (never crashing the loop) | 🟡 Intermediate |
| 4 | Hard `Budget` guardrails (tokens + wall-clock) & structured `Tracer` | 🟡 Intermediate |
| 5 | Dynamic decomposition & how the Claude Agent SDK wraps this loop | 🔴 Advanced |
| 6 | Hands-on custom tool & loop hardening exercises | 🏋️ Practice |

> **How to use this notebook:** Run cells in order. Cells with `# TODO` are exercises for you to implement.

## 0. Setup

In [ ]:
%pip install -q anthropic

import json
import os
import time
from dataclasses import dataclass, field
from getpass import getpass
from typing import Any, Callable

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

import anthropic

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"
print("Client ready with model:", MODEL)

## 1. Designing Custom Tools & Mock Ledger Data 🟢

In Anthropic's Messages API (and under the hood in Claude Agent SDK's `tool()`), a tool has three parts:
1. **`name`**: snake_case identifier
2. **`description`**: the prompt the model reads to decide *when* and *how* to call the tool (be specific about units, edge cases, and return shape!)
3. **`input_schema`**: JSON Schema defining parameters, types, descriptions, and `required` fields

Let's create a small expense ledger and two custom tools: `list_expenses` and `check_policy_limit`.

In [ ]:
EXPENSE_LEDGER = [
    {"id": "EXP-01", "employee": "Ana", "category": "meals", "amount_usd": 42.50, "receipt": True},
    {"id": "EXP-02", "employee": "Ana", "category": "meals", "amount_usd": 135.00, "receipt": False},
    {"id": "EXP-03", "employee": "Ben", "category": "lodging", "amount_usd": 290.00, "receipt": True},
    {"id": "EXP-04", "employee": "Ben", "category": "software", "amount_usd": 89.00, "receipt": False},
]

POLICY_LIMITS = {
    "meals": {"max_usd": 75.00, "receipt_required_above_usd": 25.00},
    "lodging": {"max_usd": 300.00, "receipt_required_above_usd": 0.00},
    "software": {"max_usd": 100.00, "receipt_required_above_usd": 50.00},
}

TOOLS = [
    {
        "name": "list_expenses",
        "description": "List recorded expenses. Optionally filter by employee name (exact match, e.g. 'Ana').",
        "input_schema": {
            "type": "object",
            "properties": {
                "employee": {"type": "string", "description": "Optional employee name to filter by."}
            },
            "required": [],
        },
    },
    {
        "name": "check_policy_limit",
        "description": "Return the company spending limit and receipt threshold for a given expense category (meals, lodging, software).",
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "enum": ["meals", "lodging", "software"],
                    "description": "Expense category to look up.",
                }
            },
            "required": ["category"],
        },
    },
]

## 2. Safe Tool Execution: Return Errors, Don't Crash 🟡

A critical rule from Lesson 05's Python `loop.py` and Lesson 06:
> **Tool errors are returned to Claude as `tool_result` blocks with `is_error=True`, never raised as unhandled exceptions.**

Why? If a tool raises an exception, the whole agent crashes. If it returns `is_error=True` with a clear message (e.g., `"Unknown category 'travel'. Valid categories: meals, lodging, software."`), Claude can **self-correct** on the next turn!

In [ ]:
def execute_tool(name: str, tool_input: dict[str, Any]) -> tuple[str, bool]:
    """Return (json_string_content, is_error)."""
    try:
        if name == "list_expenses":
            emp = tool_input.get("employee")
            rows = [r for r in EXPENSE_LEDGER if not emp or r["employee"].lower() == emp.lower()]
            return json.dumps({"expenses": rows}), False

        if name == "check_policy_limit":
            cat = tool_input["category"]
            if cat not in POLICY_LIMITS:
                return json.dumps({"error": f"Unknown category '{cat}'. Valid: {list(POLICY_LIMITS)}"}), True
            return json.dumps({"category": cat, **POLICY_LIMITS[cat]}), False

        return json.dumps({"error": f"Unknown tool '{name}'"}), True
    except Exception as exc:
        return json.dumps({"error": f"Tool execution failed: {exc}"}), True


print("Valid call:", execute_tool("check_policy_limit", {"category": "meals"}))
print("Error call:", execute_tool("check_policy_limit", {"category": "flights"}))

## 3. Budget Guardrails & Audit Tracer 🟡

From Lesson 05 (`Python /01-Agentic_Loop` and `02-tools-and-audit`):
- **Never** use string-matching on assistant text (like `"FINAL ANSWER" in text`) to stop a loop.
- **Never** silently truncate when a limit is hit.
- Drive control flow strictly by `response.stop_reason`, and enforce a `Budget` (token cap + wall-clock seconds) that raises `BudgetExceeded`, while recording every turn in an audit `Tracer`.

In [ ]:
class UnexpectedStopReason(Exception):
    pass


class BudgetExceeded(Exception):
    pass


@dataclass
class Budget:
    max_total_tokens: int = 12_000
    max_wall_seconds: float = 30.0
    used_tokens: int = 0
    started_at: float = field(default_factory=time.monotonic)

    def charge(self, input_tokens: int, output_tokens: int) -> None:
        self.used_tokens += input_tokens + output_tokens
        elapsed = time.monotonic() - self.started_at
        if self.used_tokens > self.max_total_tokens:
            raise BudgetExceeded(f"Token budget exceeded: {self.used_tokens} > {self.max_total_tokens}")
        if elapsed > self.max_wall_seconds:
            raise BudgetExceeded(f"Wall-clock budget exceeded: {elapsed:.1f}s > {self.max_wall_seconds}s")


@dataclass
class AuditTracer:
    events: list[dict[str, Any]] = field(default_factory=list)

    def record(self, kind: str, **payload: Any) -> None:
        self.events.append({"ts": round(time.monotonic(), 3), "kind": kind, **payload})

    def summary(self) -> None:
        for i, ev in enumerate(self.events, 1):
            print(f"{i:02d}. [{ev['kind']}] { {k: v for k, v in ev.items() if k not in ('ts', 'kind')} }")

## 4. The Production Agentic Loop 🟡

Now let's assemble `run_agent_loop(...)`:
1. Send `messages` + `tools` to `client.messages.create(...)`.
2. Charge `budget` and record the turn in `tracer`.
3. Branch strictly on `response.stop_reason`:
   - `"end_turn"` → return final text + trace.
   - `"tool_use"` → execute *every* `tool_use` block in `response.content`, append a single `user` message containing all `tool_result` blocks, and continue.
   - anything else (`"max_tokens"`, `"refusal"`, etc.) → raise `UnexpectedStopReason`.

In [ ]:
def run_agent_loop(
    user_prompt: str,
    tools: list[dict],
    executor: Callable[[str, dict], tuple[str, bool]],
    system: str = "You are a meticulous corporate expense auditor. Always verify expenses against policy limits before concluding.",
    budget: Budget | None = None,
) -> tuple[str, AuditTracer]:
    budget = budget or Budget()
    tracer = AuditTracer()
    messages: list[dict[str, Any]] = [{"role": "user", "content": user_prompt}]

    turn = 0
    while True:
        turn += 1
        resp = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=system,
            tools=tools,
            messages=messages,
        )
        budget.charge(resp.usage.input_tokens, resp.usage.output_tokens)
        tracer.record(
            "llm_turn",
            turn=turn,
            stop_reason=resp.stop_reason,
            in_tok=resp.usage.input_tokens,
            out_tok=resp.usage.output_tokens,
        )

        # Preserve full assistant content blocks (text + tool_use) in conversation history
        messages.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason == "end_turn":
            final_text = "".join(b.text for b in resp.content if b.type == "text")
            return final_text, tracer

        if resp.stop_reason == "tool_use":
            tool_results = []
            for block in resp.content:
                if block.type != "tool_use":
                    continue
                content_str, is_error = executor(block.name, block.input)
                tracer.record("tool_call", tool=block.name, input=block.input, is_error=is_error)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": content_str,
                    "is_error": is_error,
                })
            messages.append({"role": "user", "content": tool_results})
            continue

        raise UnexpectedStopReason(f"Unexpected stop_reason: {resp.stop_reason}")


answer, trace = run_agent_loop(
    "Audit Ana's expenses and flag every policy violation with exact dollar amounts.",
    tools=TOOLS,
    executor=execute_tool,
)
print("=== AUDIT TRACE ===")
trace.summary()
print("\n=== AGENT REPORT ===")
print(answer)

### 🧐 Observation
1. Notice how the agent dynamically decomposes the task: first calling `list_expenses`, discovering which categories Ana used, then calling `check_policy_limit`, and finally synthesizing the violations.
2. In the **Claude Agent SDK** (TypeScript, Lesson 05/06), `query({ prompt, options: { allowedTools, mcpServers } })` runs this exact loop under the hood, yielding typed messages until `message.type === "result"`.

## 5. Failure Modes & Fixes 🔴

| Failure Mode | Why It Happens | Fix |
|---|---|---|
| Infinite tool-call loop | Tool returns vague error; model retries identical args forever | Enforce `Budget` (tokens + wall-clock) and return actionable error messages |
| Broken multi-tool turn | Only answering the first `tool_use` block when Claude emits parallel calls | Iterate *all* `block.type == "tool_use"` blocks and send all `tool_result`s in one `user` turn |
| Dropped `tool_use_id` | API rejects `tool_result` that doesn't match the preceding assistant `tool_use.id` | Always copy `block.id` into `tool_use_id` and append `resp.content` verbatim |
| Ambiguous tool descriptions | Model picks the wrong tool or invents parameter formats | Document exact types, enums, units, and edge cases in `description` and `input_schema` |

## 6. Exercises 🏋️

### Exercise 1: Add a Third Custom Tool (`flag_expense`)
Add a new tool `flag_expense(expense_id: str, reason: str)` that appends flagged items to a `FLAGGED_LOG` list (returning `is_error=True` if `expense_id` does not exist in `EXPENSE_LEDGER`). Run `run_agent_loop` to audit *all* employees and flag violations automatically.

In [ ]:
FLAGGED_LOG: list[dict[str, str]] = []

# TODO: Add the `flag_expense` schema to EXTENDED_TOOLS
EXTENDED_TOOLS = list(TOOLS)


def extended_executor(name: str, tool_input: dict[str, Any]) -> tuple[str, bool]:
    if name == "flag_expense":
        # TODO: Validate expense_id exists in EXPENSE_LEDGER; if not, return error with is_error=True.
        # Otherwise append {"expense_id": ..., "reason": ...} to FLAGGED_LOG and return confirmation JSON.
        return json.dumps({"error": "Not implemented yet"}), True
    return execute_tool(name, tool_input)


# After implementing, run:
# report, t = run_agent_loop("Audit all employees and flag every violation.", EXTENDED_TOOLS, extended_executor)
# print("Flagged:", FLAGGED_LOG)

### Exercise 2: Test Budget Enforcement
Create a tiny `Budget(max_total_tokens=200)` and run `run_agent_loop` inside a `try / except BudgetExceeded` block to verify your loop fails loudly instead of silently truncating.

In [ ]:
def run_with_strict_budget(prompt: str, max_tokens: int) -> str:
    """Return the agent report or a clear 'BUDGET_STOPPED: ...' message."""
    # TODO: Construct Budget(max_total_tokens=max_tokens), call run_agent_loop,
    # and catch BudgetExceeded cleanly.
    return "TODO"


print(run_with_strict_budget("Audit all expenses across every employee.", max_tokens=200))

## 🏁 Summary & Key Takeaways

- **Control flow = `stop_reason`:** Continue on `"tool_use"`, return on `"end_turn"`, raise on anything else.
- **Fail loudly on budgets, gracefully on tools:** Raise `BudgetExceeded` on token/time limits; return `is_error=True` on tool errors so Claude can recover.
- **Trace every turn:** An `AuditTracer` recording LLM turns and tool calls makes debugging and evaluation trivial.
- **SDK parity:** The Claude Agent SDK (`query()`, `allowedTools`, custom MCP tools) packages this exact loop so you only supply tools, prompts, and schemas.

### What's next?
Continue to **`deep-dive-4-structured-outputs-and-context-engineering.ipynb`** to guarantee schema-valid outputs with Pydantic/JSON Schema and master the 4-stage context engineering pipeline.